[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aledamelio/ML_Pavia/blob/main/05_Regression.ipynb)

# Linear Regression

- In the classification problems discussed so far, we have considered predicting categorical target variables.

- We often encounter prediction tasks where the target variable is continuous, for example:
    - Temperature
    - Prices
    - An individual's height
    - ...

In such cases, we speak of a **Regression** problem. Specifically, a regression problem can be defined as finding a function $f(\mathbf{x})$ such that:

$$y = f(\mathbf{x})$$

where $\mathbf{x}$ represents the vector of feature variables (also called explanatory variables, predictors, or covariates), and $y$ is the target variable (also called response variable, dependent variable, or outcome variable).

If we assume that $f(\mathbf{x})$ is a linear function, we obtain a **Linear Regression** problem.

### Simple Linear Regression

- Also known as Univariate Regression
- The vector $\mathbf{x}$ has a dimensionality of $1$

The equation of a simple Linear Regression model is defined as follows:

$$y=w_{0}+w_{1} x$$

The goal is to learn the values of the coefficients (weights)—$w_{0}$ (intercept) and $w_{1}$ (slope/slope coefficient)—from a given dataset.

More generally, Linear Regression can be viewed as the problem of finding the line that "best fits" the training data, as shown in the following figure:

<img src="images/10_01.png" width="500">

This line is called the **Regression Line**. In this context, the best fit is defined as the line that minimizes the **offsets** or **residuals** (prediction errors).

### Multiple Linear Regression

- Also known as Multivariate Linear Regression
- Generalizes simple linear regression to the case where $\mathbf{x} \in \mathbb{R}^d$ with $d>1$

The equation of a Multiple Linear Regression model is defined as follows:

$$y=w_{0} x_{0}+w_{1} x_{1}+\ldots+w_{d} x_{d}=\sum_{i=0}^{d} w_{i} x_{i}=\mathbf{w}^{T} \mathbf{x}$$

- Where $x_{0} = 1$ is the dummy feature for the bias term $w_{0}$
- The goal is to learn the "best-fitting" hyperplane that passes through the data points. For $d=2$, this can be visualized as shown below:

<img src="images/10_011.png" width="500">

## Fitting a Linear Regression Model - Ordinary Least Squares (OLS)

We define more precisely what is meant by the "best" hyperplane passing through the points. As mentioned earlier, it is the hyperplane that minimizes the residuals (prediction errors).

We can then define the objective function for learning a Linear Regression model as:

$$J(\mathbf{w})=\frac{1}{2} \sum_{i=1}^{n}\left(y^{(i)}-\hat{y}^{(i)}\right)^{2} \;\;\; \text{with} \;\;\; \hat{y}^{(i)} = \mathbf{w}^{T} \mathbf{x}^{(i)}$$

- The objective function (cost function) is the **Sum of Squared Errors (SSE)**
- We aim to find the parameter vector $\mathbf{w}$ of the line (plane, hyperplane) that minimizes the SSE:

$$\arg\min_{\mathbf{w}} J(\mathbf{w})$$

## Boston Housing Dataset

To implement and test linear regression models, we use the *Boston Housing* dataset:

> "A dataset derived from information collected by the U.S. Census Service concerning housing in the area of Boston, Mass."

We load it into memory using Pandas (or alternatively, scikit-learn):

In [ ]:
import pandas as pd
import numpy as np

# Fetch the dataset directly from the original repository URL
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep=r"\s+", skiprows=22, header=None)

# Format features and target according to the repository structure
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

feature_names = [
    'CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM',
    'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'
]

# Create DataFrame
df = pd.DataFrame(data, columns=feature_names)
df['MEDV'] = target

# Display the first few rows and shape
print(df.head())
print(df.shape)

In [ ]:
df.head()

This is the description of the columns (features):

<img src="images/10_012.png" width="500">

### Exploratory Data Analysis (EDA) of the Boston Housing Dataset

One of the recommended preliminary steps before training any machine learning model is performing an **Exploratory Data Analysis (EDA)**.

We can use **Pandas** and **Seaborn** plotting methods for this purpose:

In [ ]:
pd.plotting.scatter_matrix(df[['LSTAT', 'INDUS', 'NOX', 'RM', 'MEDV']], alpha=0.5, figsize=(15,10));

This pair plot allows us to visualize the pairwise relationships between the selected variables ('LSTAT', 'INDUS', 'NOX', 'RM', 'MEDV') across the entire DataFrame, alongside histograms showing their univariate distributions.

Focusing on the target variable 'MEDV' (bottom row):

- There is an approximately linear relationship between 'MEDV' and the feature 'RM'
- There is a clear (albeit non-linear) relationship between 'MEDV' and 'LSTAT'
- The other two variables ('INDUS' and 'NOX') do not appear to be particularly informative

We can perform a more quantitative evaluation by computing the correlation matrix:

- Pearson's correlation coefficient is calculated for each pair of variables
- It quantifies the degree of linear correlation between variables


In [ ]:
#corr = df[['LSTAT', 'INDUS', 'NOX', 'RM', 'MEDV']].corr()
corr = df.corr()
corr.style.background_gradient(cmap='coolwarm').format(precision=2)

## OLS for Linear Regression in Python

Consider the objective function previously defined (**SSE**):

$$J(\mathbf{w})=\frac{1}{2} \sum_{i=1}^{n}\left(y^{(i)}-\hat{y}^{(i)}\right)^{2} \;\;\; \text{with} \;\;\; \hat{y}^{(i)} = \mathbf{w}^{T} \mathbf{x}^{(i)}$$

We note that this is the same cost function used for ADALINE (Adaptive Linear Neuron).

- In essence, OLS can therefore be associated with the learning procedure of ADALINE
- The difference lies in the fact that while the target values ($y^{(i)}$) in ADALINE represented discrete class labels (e.g., $-1, +1$), in OLS they represent continuous real values

We can use the same implementation as ADALINE, simply removing the **step function** during the prediction phase:

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)

class RegressioneLineare(object):
    
    def __init__(self, eta=0.001, n_iter=50, random_state=123):
        self.eta = eta
        self.n_iter = n_iter
        np.random.seed(random_state)

    def fit(self, X, y, method='GD'):
        
        if method=='GD':
        
            self.w_ = np.random.normal(loc=0.0, scale=0.01, size=1 + X.shape[1])
            self.cost_ = []
            self.norms_ = []

            for i in range(self.n_iter):
                self.cost_.append(self._update_weights(X, y))
                self.norms_.append(np.linalg.norm(self.w_))
        else:

            X = np.hstack([np.ones([X.shape[0],1]), X])
            self.w_ = np.linalg.pinv(X) @ y
            
        return self

    def integrate(self, X):
        """Calculate net input"""
        return np.dot(X, self.w_[1:]) + self.w_[0]
    
    def _update_weights(self, X, y):
        """Apply Adaline learning rule to update the weights"""
        output = self.integrate(X)
        errors = (y - output)        
        self.w_[1:] += self.eta * X.T.dot(errors)
        self.w_[0] += self.eta * errors.sum()
        cost = (errors**2).sum() / 2.0
        return cost

    def predict(self, X):
        """Return prediction"""
        return self.integrate(X)
        #Con la step function (ADALINE)...
        #return np.where(self.integrate(X) >= 0.0, 1, -1)

Note that, exactly as with ADALINE, learning can take place through two main approaches:

- **Iteratively**, using Gradient Descent (GD):

$$w_{j} := w_{j} + \Delta w_{j}$$

with:

$$\Delta w_{j} = \eta \sum_{i}\left(y^{(i)} - \hat{y}^{(i)}\right) x_{j}^{(i)}$$

- **In closed form**, using the Moore-Penrose pseudo-inverse (Normal Equations):

$$\boldsymbol{w} = \boldsymbol{X}^{\dagger}\boldsymbol{y}$$

with:

$$\boldsymbol{X}^{\dagger} = \left(\boldsymbol{X}^{T} \boldsymbol{X}\right)^{-1} \boldsymbol{X}^{T}$$

### Linear Regression on the Boston Housing Dataset

In [ ]:
X = df[['RM']].values
y = df[['MEDV']].values

In [ ]:
from sklearn.preprocessing import StandardScaler

sc_x = StandardScaler()
sc_y = StandardScaler()
X_std = sc_x.fit_transform(X)
y_std = sc_y.fit_transform(y).ravel()

#### Iterative method (GD)

In [ ]:
import matplotlib.pyplot as plt

lr = RegressioneLineare(eta=0.001, n_iter=10)
lr.fit(X_std, y_std, method='GD')

plt.figure(figsize=(15,5))
plt.subplot(1,1,1)
plt.plot(range(lr.n_iter), lr.cost_)
plt.ylabel('ESS')
plt.grid()
plt.xlabel('Epoch')

plt.show()

#### Closed form

In [ ]:
lr_closed = RegressioneLineare()
lr_closed.fit(X_std, y_std, method='CF');

In [ ]:
lr.w_

In [ ]:
lr_closed.w_

### Let's inspect the results

We visualize the fitting of the regression line on the data

In [ ]:
def regplot(X, y, model):
    plt.scatter(X, y, c='steelblue', edgecolor='white', s=70)
    plt.plot(X, model.predict(X), color='black', lw=2)   
    
regplot(X_std, y_std, lr)
plt.xlabel('Average number of rooms [RM] (standardized)')
plt.ylabel('Price in $1000s [MEDV] (standardized)')

plt.show()

#### Comparison with scikit-learn

In [ ]:
from sklearn.linear_model import LinearRegression

slr = LinearRegression()
slr.fit(X_std, y_std)
y_pred = slr.predict(X)
print('Slope: %.3f' % slr.coef_[0])
print('Intercept: %.3f' % slr.intercept_)

plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
regplot(X_std, y_std, lr)
plt.xlabel('Average number of rooms [RM] (standardized)')
plt.ylabel('Price in $1000s [MEDV] (standardized)')
plt.title('RegressionLine')

plt.subplot(1,2,2)
regplot(X_std, y_std, slr)
plt.xlabel('Average number of rooms [RM] (standardized)')
plt.ylabel('Price in $1000s [MEDV] (standardized)')
plt.title('sk-learn')
plt.show()

## Residual Plots

A residual plot is a diagnostic tool used to evaluate the performance of a regression model and verify its underlying assumptions.

It displays the predicted (fitted) values on the $x$-axis and the residuals on the $y$-axis, where a residual is defined as the difference between the predicted value and the actual target value:

$$\text{residual}^{(i)} = \hat{y}^{(i)} - y^{(i)}$$

---

> **Key Diagnostic Checks:**
> - **Zero Mean & Homoscedasticity:** Residuals should be randomly scattered around the horizontal line $y = 0$ with roughly constant variance across all predicted values.
> - **Non-linearity:** Systematic patterns or curves (e.g., U-shapes) indicate that a linear model fails to capture non-linear trends in the data.
> - **Outliers:** Points located unusually far from the zero line represent potential outliers or high-leverage instances.

In [ ]:
def residual_plot(X, y, model, ylim=[-35, 45], title=None):
    y_pred = model.predict(X)
    plt.scatter(y_pred,  y_pred - y, c='steelblue', marker='o', edgecolor='k',)
    plt.xlabel('Predicted values')
    plt.ylabel('Residuals')
    plt.hlines(y=0, xmin=np.min(y_pred), xmax=np.max(y_pred), color='black', lw=2)
    plt.ylim(ylim)
    plt.title(title)
    plt.grid()

X_rm = df[['RM']].values
X_all = df.iloc[:, :-1].values
y = df['MEDV'].values

#Regressione Lineare Semplice
slr_rm = LinearRegression()
slr_rm.fit(X_rm, y)
y_pred_rm = slr_rm.predict(X_rm)

#Regressione Lineare Multipla
slr_all = LinearRegression()
slr_all.fit(X_all, y)
y_pred_all = slr_all.predict(X_all)

plt.figure(figsize=(15,5))

plt.subplot(1,2,1)
residual_plot(X_rm, y, slr_rm, ylim=[-35, 30], title='Only RM')

plt.subplot(1,2,2)
residual_plot(X_all, y, slr_all, ylim=[-35, 30], title='All Features')

plt.show()

# Polynomial Regression

The assumption of linearity between features and the target variable can be overly restrictive in many real-world scenarios.

One way to account for non-linear relationships within a regression model is to include polynomial terms:

$$y=w_{0}+w_{1} x+w_{2} x^{2}+\ldots+w_{d} x^{d}$$

where $d$ represents the degree of the polynomial. Note that despite incorporating terms of degree $d$, the model remains linear with respect to the parameters $\mathbf{w}$. Consequently, we can still treat this as a Multiple Linear Regression problem over an expanded feature space.

### Polynomial Regression in Python

We can use the `PolynomialFeatures` class from `scikit-learn` to transform our original features through polynomial expansion.

Consider the following example:

In [ ]:
X = np.array([258.0, 270.0, 294.0, 
              320.0, 342.0, 368.0, 
              396.0, 446.0, 480.0, 586.0])\
             [:, np.newaxis]

y = np.array([236.4, 234.4, 252.8, 
              298.6, 314.2, 342.2, 
              360.8, 368.0, 391.2,
              390.8])

plt.scatter(X,y)
plt.grid()
plt.show()

We expand the features $X$ using a polynomial of degree $2$:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
quadratic = PolynomialFeatures(degree=2)

X_quad = quadratic.fit_transform(X)

print(X)
print(X_quad)

We use a Simple Linear Regression model on the original dataset and a Multiple Linear Regression model on the expanded features:

In [ ]:
lr = LinearRegression()
pr = LinearRegression()

# fit linear features
lr.fit(X, y)
X_fit = np.arange(250, 600, 10)[:, np.newaxis]
y_lin_fit = lr.predict(X_fit)

# fit quadratic features
pr.fit(X_quad, y)
y_quad_fit = pr.predict(quadratic.fit_transform(X_fit))

# plot results
plt.scatter(X, y, label='Training points')
plt.plot(X_fit, y_lin_fit, label='Linear fit', linestyle='--')
plt.plot(X_fit, y_quad_fit, label='Quadratic fit')
plt.xlabel('Explanatory variable')
plt.ylabel('Predicted or known target values')
plt.legend(loc='upper left')

plt.grid()

plt.tight_layout()
plt.show()

#### Residual Plot

In [ ]:
def residual_plot(X, y, model, ylim=[-35, 45], title=None):
    y_pred = model.predict(X)
    plt.scatter(y_pred,  y_pred - y, c='steelblue', marker='o', edgecolor='k',)
    plt.xlabel('Predicted values')
    plt.ylabel('Residuals')
    plt.hlines(y=0, xmin=np.min(y_pred), xmax=np.max(y_pred), color='black', lw=2)
    plt.ylim(ylim)
    #plt.xlim([np.min(y), np.max(y)])
    plt.title(title)
    plt.grid()

plt.figure(figsize=(15,5))
plt.subplot(1,2,1)
residual_plot(X, y, lr)
plt.subplot(1,2,2)
residual_plot(X_quad, y, pr)
plt.show()


## Modeling Non-Linear Relationships in the Boston Housing Dataset

Consider the scatter plots generated during our Exploratory Data Analysis (EDA) on the Boston Housing dataset:

In [ ]:
pd.plotting.scatter_matrix(df[['LSTAT', 'RM', 'MEDV']], alpha=0.5, figsize=(15,10));

We observe that while the relationship between **`RM`** and the target variable **`MEDV`** is approximately linear, the relationship between **`LSTAT`** and **`MEDV`** exhibits marked non-linearity.

We model the relationship between these two variables using Polynomial Regression with quadratic ($d=2$) and cubic ($d=3$) terms:

In [ ]:
X = df[['LSTAT']].values
y = df['MEDV'].values

# create quadratic features
quadratic = PolynomialFeatures(degree=2)
# create cubic features
cubic = PolynomialFeatures(degree=3)

X_quad = quadratic.fit_transform(X)
X_cubic = cubic.fit_transform(X)

# fit features
X_fit = np.arange(X.min(), X.max(), 1)[:, np.newaxis]

regr_lin = LinearRegression()
regr_lin = regr_lin.fit(X, y)
y_lin_fit = regr_lin.predict(X_fit)

regr_quad = LinearRegression()
regr_quad = regr_quad.fit(X_quad, y)
y_quad_fit = regr_quad.predict(quadratic.fit_transform(X_fit))

regr_cub = LinearRegression()
regr_cub = regr_cub.fit(X_cubic, y)
y_cubic_fit = regr_cub.predict(cubic.fit_transform(X_fit))

# plot results
plt.figure(figsize=(12,8))
plt.scatter(X, y, label='Training points', color='lightgray')
plt.plot(X_fit, y_lin_fit, 
         label='Linear (d=1)', 
         color='blue', 
         lw=2, 
         linestyle=':')

plt.plot(X_fit, y_quad_fit, 
         label='Quadratic (d=2)',
         color='red', 
         lw=2,
         linestyle='-')

plt.plot(X_fit, y_cubic_fit, 
         label='Cubic (d=3)',
         color='green', 
         lw=2, 
         linestyle='--')

plt.xlabel('% lower status of the population [LSTAT]')
plt.ylabel('Price in $1000s [MEDV]')
plt.legend(loc='upper right')

plt.show()

In [ ]:
plt.figure(figsize=(15,5))
plt.subplot(1,3,1)
residual_plot(X, y, regr_lin, ylim=[-35, 30], title='Linear')
plt.subplot(1,3,2)
residual_plot(X_quad, y, regr_quad, ylim=[-35, 30], title='Quadratic')
plt.subplot(1,3,3)
residual_plot(X_cubic, y, regr_cub, ylim=[-35, 30], title='Cubic')
plt.show()

## Other features transformations

Polynomial features are not always the optimal choice for modeling non-linear relationships. For instance, observing the relationship between **`LSTAT`** and **`MEDV`**, it is easy to note that the trend closely resembles a negative exponential function:

In [ ]:
X = df[['LSTAT']].values
y = df['MEDV'].values

xx = np.linspace(np.min(X), np.max(X), 100)
def expFun(x, m, t, b):
    return m * np.exp(-t * x) + b

plt.scatter(X, y, color='lightgray')
plt.plot(xx, expFun(xx, 41.5, 0.14, 12.4))
plt.show()


We note that by applying a logarithmic transformation (natural logarithm) to the variable **`LSTAT`**, the data points become more linearly distributed:

In [ ]:
def regplot(X, y, model, title=None):
    plt.scatter(X, y, c='steelblue', edgecolor='white', s=70)
    X_fit = np.arange(X.min(), X.max(), 1)[:, np.newaxis]
    plt.plot(X_fit, model.predict(X_fit), color='black', lw=2) 
    plt.title(title)

X = df[['LSTAT']].values
y = df['MEDV'].values
X_log = np.log(X)

regr = LinearRegression()
regr = regr.fit(X_log, y)

#Plot
regplot(X_log, y, regr, title=None)
plt.xlabel('log(% lower status of the population [LSTAT])')
plt.ylabel(r'$Price \; in \; \$1000s \; [MEDV]$')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(15,10))
plt.subplot(2,2,1)
residual_plot(X, y, regr_lin, ylim=[-35, 30], title='Lineare')
plt.subplot(2,2,2)
residual_plot(X_quad, y, regr_quad, ylim=[-35, 30], title='Quadratica')
plt.subplot(2,2,3)
residual_plot(X_cubic, y, regr_cub, ylim=[-35, 30], title='Cubica')
plt.subplot(2,2,4)
residual_plot(X_log, y, regr, ylim=[-35, 30], title='Natural Log')
plt.show()

# Other Regression Methods in scikit-learn

Let's explore an alternative approach to Linear Regression for predicting continuous target values: **k-NN Regression**.

### k-NN Regression

This method generalizes the k-Nearest Neighbors classification algorithm to continuous target prediction problems while retaining its core principle:

- Choose a parameter $k$ and a distance metric (e.g., Euclidean distance)
- Identify the $k$ nearest training instances to the query point for which a prediction is needed
- Predict the target value by averaging the target values of these $k$ neighboring observations

<Image src="image_agent_tag_17953760009882767149" alt="Illustration of k-NN Regression showing neighbor selection and target averaging" caption="k-NN Regression workflow" />


In [ ]:
def regplot(X, y, model, title=None):
    plt.scatter(X, y, c='steelblue', edgecolor='white', s=70)
    X_fit = np.arange(X.min(), X.max(), 1)[:, np.newaxis]
    plt.plot(X_fit, model.predict(X_fit), color='black', lw=2) 
    plt.title(title)
    
from sklearn import neighbors

X = df[['LSTAT']].values
y = df['MEDV'].values

#Pesi uniformi
knn_reg_u = neighbors.KNeighborsRegressor(n_neighbors=50, weights='uniform')
knn_reg_u.fit(X, y)
#Pesi inversamente proporzionali alla distanza
knn_reg_d = neighbors.KNeighborsRegressor(n_neighbors=50, weights='distance')
knn_reg_d.fit(X, y)

plt.figure(figsize=(15,10))
plt.subplot(2,2,1)
regplot(X, y, knn_reg_u, title='k=50, Unif weigths')
plt.xlabel('lower status of the population [LSTAT]')
plt.ylabel(r'$Price \; in \; \$1000s \; [MEDV]$')
plt.tight_layout()
plt.subplot(2,2,2)
regplot(X, y, knn_reg_d, title='k=50, Dist weigths')
plt.xlabel('lower status of the population [LSTAT]')
plt.ylabel(r'$Price \; in \; \$1000s \; [MEDV]$')
plt.tight_layout()
plt.subplot(2,2,3)
residual_plot(X, y, knn_reg_u, ylim=[-35, 30])
plt.subplot(2,2,4)
residual_plot(X, y, knn_reg_d, ylim=[-35, 30])

plt.show()

### The role of k...

In [ ]:
plt.figure(figsize=(15,10))
ks = [1, 10, 20, 50, 100, 200]

for i,k in enumerate(ks):
    plt.subplot(2,3,i+1)
    knn_reg = neighbors.KNeighborsRegressor(n_neighbors=k)
    knn_reg.fit(X, y)

    regplot(X, y, knn_reg, title='k='+str(k))

    plt.xlabel('LSTAT')
    plt.ylabel('Price')
    plt.tight_layout()
plt.show()